# Flow: proprioception-tuned CNN vs. random-projection CNN, 128-dim latents

Compares the proprioception-tuned CNN sensor-processing models against the
training-free random-projection CNN models, at a 128-dim latent size, for both
the VGG19 and ResNet50 backbones.

Like `Flow_VisualProprioception.ipynb`, this redirects `Config` to an external,
git-decoupled `expruns`/`results` directory via `visproprio_helper.external_setup`,
imports its demonstration data with `import_demopack`, and (re)trains everything
it needs inside that fresh directory.

Unlike `Flow_VisualProprioception.ipynb`, it does not sweep over model families
(VAE, ViT) or latent sizes. The proprioception-tuned sensor-processing models
are `create_exprun_variant` variants of the existing curated `vgg19_128` /
`resnet50_128` exp/runs, so all their tuned hyperparameters are preserved and
only the demopack-derived training data is substituted. The random-projection
sensor-processing models are training-free and are used exactly as they are.

The comparison is evaluated on the demopack's `vp_testing` group, which is held
out of both the sensor-processing and the regressor training sets. The
`vp_validation` group overlaps `sp_training`, which would favour the
proprioception-tuned models over the random-projection ones.

Steps:
1. Set up the external flow directory and copy in the exp/run families this
   flow needs (including `sensorprocessing_random_projection_cnn`, which
   `external_setup` does not copy on its own).
2. Import the demopack and derive the training/evaluation data lists.
3. Create the two proprioception-tuned sensor-processing variants and train them.
4. Generate the four `visual_proprioception` exp/runs and train them.
5. Generate and run the comparison exp/run.


In [1]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import yaml
import tqdm
import papermill
import visproprio_helper
from demonstration.demopack import import_demopack, group_chooser_sp_vp_standard


## Set up the external flow directory

Creates `<flows_path>/<flow_name>/{expruns,results}` and points `Config` at it,
copying in the exp/run families the flow depends on. `external_setup` does not
yet know about `sensorprocessing_random_projection_cnn`, so it is copied here
explicitly.


In [2]:
flow_name = "PtunVsRandProj_128_01"

# Use exist-ok not to re-run previously successfully run models, if this
# flow is executed again.
creation_style = "exist-ok"

expruns_path, results_path = visproprio_helper.external_setup(
    flow_name, pathlib.Path(Config()["flows_path"]).expanduser()
)

Config().copy_experiment("sensorprocessing_random_projection_cnn")


***ExpRun**: Loading pointer config file:
	/Users/lboloni/.config/BerryPicker/mainsettings.yaml
***ExpRun**: Loading machine-specific config file:
	/Users/lboloni/Google Drive/My Drive/LotziStudy/Code/PackageTracking/BerryPicker/settings/settings-szenes.yaml
***ExpRun**: Using torch device: cpu
***Path for external experiments:
/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/PtunVsRandProj_128_01/expruns
***Path for external data:
/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/PtunVsRandProj_128_01/results
***ExpRun**: Experiment config path changed to /Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/PtunVsRandProj_128_01/expruns
***ExpRun**: Experiment data path changed to /Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/PtunVsRandProj_128_01/results
***ExpRun**: Experiment robot_al5d copied to
/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/PtunVsRandProj_128_01/expruns/robot_al5d
***ExpRun**: Experiment demonstration copied to
/Users/lboloni/Do

## Import the demopack

`import_demopack` copies the demonstrations into the flow's results directory,
renaming them by group, and returns the group membership. The sensor-processing
models train on `sp_training`, the regressors on `vp_training`, and the
comparison evaluates on `vp_testing`.


In [3]:
demopack_name = "random-both-cameras-video"
demonstration_cam = "dev2"

demopack_path = pathlib.Path(Config()["demopacks_path"]).expanduser() / demopack_name
selection = import_demopack(demopack_path, group_chooser_sp_vp_standard)

def demo_data(group):
    return [[demopack_name, demo, demonstration_cam] for demo in selection[group]]

sp_training_data = demo_data("sp_training")
sp_validation_data = demo_data("sp_validation")
vp_training_data = demo_data("vp_training")
vp_eval_data = demo_data("vp_testing")


## Parameters

The four models being compared, and the sensor-processing variants two of them
are built on.


In [4]:
latent_size = 128
epochs_vp = 1000

# Variants of the curated proprioception-tuned CNN exp/runs, with the
# demopack-derived training data substituted in.
sp_variants = [
    ("vgg19_128", "vgg19_128_flow"),
    ("resnet50_128", "resnet50_128_flow"),
]

vp_runs = [
    {"run": "vp_ptun_vgg19_128", "name": "ptun-vgg19-128",
     "sp_experiment": "sensorprocessing_propriotuned_cnn",
     "sp_run": "vgg19_128_flow",
     "sensor_processing": "VGG19ProprioTunedSensorProcessing"},
    {"run": "vp_ptun_resnet50_128", "name": "ptun-resnet50-128",
     "sp_experiment": "sensorprocessing_propriotuned_cnn",
     "sp_run": "resnet50_128_flow",
     "sensor_processing": "ResNetProprioTunedSensorProcessing"},
    {"run": "vp_randproj_vgg19_128", "name": "randproj-vgg19-128",
     "sp_experiment": "sensorprocessing_random_projection_cnn",
     "sp_run": "vgg19_rademacher_128",
     "sensor_processing": "RandomProjectionCNNSensorProcessing"},
    {"run": "vp_randproj_resnet50_128", "name": "randproj-resnet50-128",
     "sp_experiment": "sensorprocessing_random_projection_cnn",
     "sp_run": "resnet50_rademacher_128",
     "sensor_processing": "RandomProjectionCNNSensorProcessing"},
]

compare_run_name = "comp_ptun_vs_randproj_128"


## Create and queue the proprioception-tuned sensor-processing variants

`VGG19ProprioTunedSensorProcessing` / `ResNetProprioTunedSensorProcessing`
require a trained checkpoint, so these have to run before any
`visual_proprioception` step that depends on them. The random-projection models
are training-free and need no such step.


In [5]:
entries = []
for sp_run, variant_run in sp_variants:
    Config().create_exprun_variant(
        "sensorprocessing_propriotuned_cnn", sp_run,
        {"training_data": sp_training_data,
         "validation_data": sp_validation_data},
        new_run_name=variant_run)
    entries.append({
        "notebook": "sensorprocessing/Train_ProprioTuned_CNN.ipynb",
        "experiment": "sensorprocessing_propriotuned_cnn",
        "run": variant_run,
    })


***ExpRun**: Configuration for exp/run: sensorprocessing_propriotuned_cnn/vgg19_128 successfully loaded
***ExpRun**: Exp/run variant sensorprocessing_propriotuned_cnn/vgg19_128_flow created in /Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/PtunVsRandProj_128_01/expruns/sensorprocessing_propriotuned_cnn/vgg19_128_flow.yaml
***ExpRun**: Configuration for exp/run: sensorprocessing_propriotuned_cnn/resnet50_128 successfully loaded
***ExpRun**: Exp/run variant sensorprocessing_propriotuned_cnn/resnet50_128_flow created in /Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/PtunVsRandProj_128_01/expruns/sensorprocessing_propriotuned_cnn/resnet50_128_flow.yaml


## Generate and queue the `visual_proprioception` exp/runs

Written into the external `expruns_path`, in the same format as the
hand-authored `vp_ptun_vgg19_128.yaml` / `vp_aruco_128.yaml`, but with the
demopack-derived data lists rather than the literal demo names those files
inherit from `_defaults_visual_proprioception.yaml`.


In [6]:
def generate_vp_run(spec):
    val = {
        "input-to-notebook": [
            "visual_proprioception/Train_VisualProprioception.ipynb",
            "visual_proprioception/Verify_VisualProprioception.ipynb",
        ],
        "name": spec["name"],
        "output_size": 6,
        "proprioception_training_task": "proprio_regressor_training",
        "proprioception_testing_task": "proprio_regressor_validation",
        "encoding_size": latent_size,
        "regressor_hidden_size_1": 64,
        "regressor_hidden_size_2": 64,
        "loss": "MSE",
        "epochs": epochs_vp,
        "training_data": vp_training_data,
        "validation_data": vp_eval_data,
        "sensor_processing": spec["sensor_processing"],
        "sp_experiment": spec["sp_experiment"],
        "sp_run": spec["sp_run"],
    }
    path = pathlib.Path(
        Config().get_exprun_path(), "visual_proprioception", spec["run"] + ".yaml"
    )
    with open(path, "w") as f:
        yaml.dump(val, f)
    return {
        "notebook": "visual_proprioception/Train_VisualProprioception.ipynb",
        "experiment": "visual_proprioception",
        "run": spec["run"],
    }


entries += [generate_vp_run(spec) for spec in vp_runs]


## Generate and queue the comparison exp/run

Written into `<expruns_path>/visual_proprioception_collections/`, listing all
four runs for `Compare_VisualProprioception.ipynb` to plot side by side.


In [7]:
def generate_vp_compare(run_name, tocompare):
    val = {
        "input-to-notebook": ["visual_proprioception/Compare_VisualProprioception.ipynb"],
        "name": run_name,
        "tocompare": tocompare,
        "proprioception_training_task": "proprio_regressor_training",
        "proprioception_testing_task": "proprio_regressor_validation",
    }
    path = pathlib.Path(
        Config().get_exprun_path(), "visual_proprioception_collections", run_name + ".yaml"
    )
    with open(path, "w") as f:
        yaml.dump(val, f)
    return {
        "notebook": "visual_proprioception/Compare_VisualProprioception.ipynb",
        "experiment": "visual_proprioception_collections",
        "run": run_name,
    }


entries.append(generate_vp_compare(compare_run_name, [spec["run"] for spec in vp_runs]))
entries


[{'notebook': 'sensorprocessing/Train_ProprioTuned_CNN.ipynb',
  'experiment': 'sensorprocessing_propriotuned_cnn',
  'run': 'vgg19_128_flow'},
 {'notebook': 'sensorprocessing/Train_ProprioTuned_CNN.ipynb',
  'experiment': 'sensorprocessing_propriotuned_cnn',
  'run': 'resnet50_128_flow'},
 {'notebook': 'visual_proprioception/Train_VisualProprioception.ipynb',
  'experiment': 'visual_proprioception',
  'run': 'vp_ptun_vgg19_128'},
 {'notebook': 'visual_proprioception/Train_VisualProprioception.ipynb',
  'experiment': 'visual_proprioception',
  'run': 'vp_ptun_resnet50_128'},
 {'notebook': 'visual_proprioception/Train_VisualProprioception.ipynb',
  'experiment': 'visual_proprioception',
  'run': 'vp_randproj_vgg19_128'},
 {'notebook': 'visual_proprioception/Train_VisualProprioception.ipynb',
  'experiment': 'visual_proprioception',
  'run': 'vp_randproj_resnet50_128'},
 {'notebook': 'visual_proprioception/Compare_VisualProprioception.ipynb',
  'experiment': 'visual_proprioception_collec

## Run the flow

Runs the queued notebooks in order via papermill: the two proprioception-tuned
CNN sensor-processing models, then the four `visual_proprioception` regressors,
then the comparison. `expruns_path` and `results_path` are passed to every
sub-notebook so each one resolves against this same external directory rather
than its own internal default.


In [8]:
for entry in tqdm.tqdm(entries):
    notebook_path = pathlib.Path("..", entry["notebook"])
    output_filename = f"{notebook_path.stem}_{entry['experiment']}_{entry['run']}_output{notebook_path.suffix}"
    output_path = pathlib.Path(results_path, output_filename)
    params = {
        "experiment": entry["experiment"],
        "run": entry["run"],
        "creation_style": creation_style,
        "expruns_path": expruns_path.as_posix(),
        "results_path": results_path.as_posix(),
    }
    print(f"*** Running {entry['notebook']} : {entry['experiment']}/{entry['run']}")
    try:
        papermill.execute_notebook(
            notebook_path,
            output_path.absolute(),
            cwd=notebook_path.parent,
            parameters=params,
            kernel_name="berrypicker",
        )
    except Exception as e:
        print(f"There was an exception {e}")


  0%|          | 0/7 [00:00<?, ?it/s]

*** Running sensorprocessing/Train_ProprioTuned_CNN.ipynb : sensorprocessing_propriotuned_cnn/vgg19_128_flow


/Users/lboloni/Documents/Develop/VirtualEnvs/BerryPicker/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 14%|█▍        | 1/7 [29:38<2:57:53, 1778.87s/it]

*** Running sensorprocessing/Train_ProprioTuned_CNN.ipynb : sensorprocessing_propriotuned_cnn/resnet50_128_flow


 29%|██▊       | 2/7 [2:34:22<7:07:53, 5134.71s/it]

...

 71%|███████▏  | 5/7 [2:41:34<39:47, 1193.92s/it]  

*** Running visual_proprioception/Train_VisualProprioception.ipynb : visual_proprioception/vp_randproj_resnet50_128


 86%|████████▌ | 6/7 [2:43:14<13:42, 822.07s/it] 

*** Running visual_proprioception/Compare_VisualProprioception.ipynb : visual_proprioception_collections/comp_ptun_vs_randproj_128


100%|██████████| 7/7 [2:43:21<00:00, 1400.19s/it]
